# Entrenamiento de CNN para Clasificación de Quemaduras y Cortadas
## Usando GPU en Google Colab

Este notebook está optimizado para entrenar el modelo mejorado con GPU.


## 1. Configurar GPU

Ve a **Runtime → Change runtime type → Hardware accelerator → GPU** y selecciona **T4 GPU** o **A100 GPU**


In [ ]:
# Verificar que GPU esté disponible
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

# Configurar para usar GPU
if tf.config.list_physical_devices('GPU'):
    print("✅ GPU detectada y lista para usar")
    # Configurar crecimiento de memoria GPU
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print("✅ Configuración de memoria GPU completada")
        except RuntimeError as e:
            print(e)
else:
    print("⚠️ No se detectó GPU. El entrenamiento será más lento.")


## 2. Instalar dependencias


In [ ]:
# Instalar dependencias necesarias
!pip install tensorflow pillow matplotlib numpy


## 3. Subir datos

Tienes dos opciones:
1. **Subir carpeta completa**: Usa el panel izquierdo de Colab para subir la carpeta `data`
2. **Usar Google Drive**: Monta tu Google Drive y copia los datos


In [ ]:
# OPCIÓN 1: Subir archivos manualmente
# Usa el panel izquierdo de Colab → Files → Upload para subir la carpeta 'data'
# Luego ejecuta esta celda para verificar
import os
if os.path.exists('data/train') and os.path.exists('data/val'):
    print("✅ Datos encontrados")
    print(f"Imágenes de entrenamiento: {sum([len(files) for r, d, files in os.walk('data/train')])}")
    print(f"Imágenes de validación: {sum([len(files) for r, d, files in os.walk('data/val')])}")
else:
    print("⚠️ No se encontraron los datos. Por favor sube la carpeta 'data'")
    print("Estructura esperada:")
    print("  data/")
    print("    train/")
    print("      Brazo/")
    print("        cortes/")
    print("        quemaduras/")
    print("      Pierna/")
    print("        cortes/")
    print("        quemaduras/")
    print("    val/")
    print("      ...")


In [ ]:
# OPCIÓN 2: Usar Google Drive
# Descomenta y ejecuta si prefieres usar Google Drive

# from google.colab import drive
# drive.mount('/content/drive')

# # Copiar datos desde Drive a Colab
# import shutil
# if os.path.exists('/content/drive/MyDrive/IA-Convolucional/data'):
#     if os.path.exists('data'):
#         shutil.rmtree('data')
#     shutil.copytree('/content/drive/MyDrive/IA-Convolucional/data', 'data')
#     print("✅ Datos copiados desde Google Drive")
# else:
#     print("⚠️ No se encontró la carpeta en Drive. Verifica la ruta.")


## 4. Código del modelo mejorado

Este código está optimizado para GPU y Colab


In [ ]:
# Importar librerías
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, 
    RandomFlip, RandomRotation, BatchNormalization,
    GlobalAveragePooling2D, Activation
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
from PIL import Image
import os
import numpy as np

print("✅ Librerías importadas correctamente")


In [ ]:
# Función para limpiar imágenes
def limpiar_y_convertir_carpeta(carpeta):
    """Limpia y convierte imágenes a formato RGB/JPG"""
    archivos_procesados = 0
    archivos_eliminados = 0
    
    for root, dirs, files in os.walk(carpeta):
        for archivo in files:
            ruta_archivo = os.path.join(root, archivo)
            try:
                img = Image.open(ruta_archivo)
                img.verify()
                img = Image.open(ruta_archivo).convert('RGB')
                nuevo_nombre = os.path.splitext(ruta_archivo)[0] + ".jpg"
                img.save(nuevo_nombre, "JPEG")
                if ruta_archivo != nuevo_nombre:
                    os.remove(ruta_archivo)
                archivos_procesados += 1
            except Exception as e:
                print(f"⚠️ Archivo inválido eliminado: {ruta_archivo}")
                try:
                    os.remove(ruta_archivo)
                    archivos_eliminados += 1
                except:
                    pass
    
    return archivos_procesados, archivos_eliminados

# Limpiar imágenes
print("🧹 Limpiando y convirtiendo imágenes de entrenamiento...")
proc_train, elim_train = limpiar_y_convertir_carpeta("data/train")
print(f"✅ Procesadas: {proc_train}, Eliminadas: {elim_train}")

print("🧹 Limpiando y convirtiendo imágenes de validación...")
proc_val, elim_val = limpiar_y_convertir_carpeta("data/val")
print(f"✅ Procesadas: {proc_val}, Eliminadas: {elim_val}")


In [ ]:
# Configurar generadores de datos con aumento mejorado
IMG_SIZE = 224  # Tamaño optimizado para GPU
BATCH_SIZE = 32  # Puedes aumentar a 64 o 128 si tienes GPU potente

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    shear_range=0.2,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

# Rutas de datos
TRAIN_DIR = "data/train"
VAL_DIR = "data/val"

# Crear generadores
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="sparse"
)

print(f"✅ Clases encontradas: {train_generator.class_indices}")
print(f"✅ Imágenes de entrenamiento: {train_generator.samples}")
print(f"✅ Imágenes de validación: {val_generator.samples}")


In [ ]:
# Construir modelo CNN mejorado
model = Sequential([
    # Data augmentation en la capa del modelo
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    
    # Bloque 1
    Conv2D(32, (3, 3), padding='same', kernel_regularizer=l2(0.001), input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(32, (3, 3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2, 2),
    Dropout(0.25),
    
    # Bloque 2
    Conv2D(64, (3, 3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(64, (3, 3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2, 2),
    Dropout(0.25),
    
    # Bloque 3
    Conv2D(128, (3, 3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(128, (3, 3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2, 2),
    Dropout(0.25),
    
    # Bloque 4
    Conv2D(256, (3, 3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(256, (3, 3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2, 2),
    Dropout(0.25),
    
    # Global Average Pooling
    GlobalAveragePooling2D(),
    
    # Capas densas
    Dense(512, kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),
    
    Dense(256, kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),
    
    # Capa de salida
    Dense(2, activation='softmax')
])

# Compilar modelo
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Mostrar resumen del modelo
model.summary()

# Contar parámetros
total_params = model.count_params()
print(f"\n📊 Total de parámetros: {total_params:,}")


In [ ]:
# Configurar callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'modelo_quemaduras_cortadas.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)

print("✅ Callbacks configurados")


In [ ]:
# Entrenar modelo
print("🚀 Iniciando entrenamiento con GPU...")
print(f"📊 Batch size: {BATCH_SIZE}")
print(f"🖼️ Tamaño de imagen: {IMG_SIZE}x{IMG_SIZE}")
print(f"📈 Épocas máximas: 200")

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=200,
    callbacks=[early_stop, checkpoint, reduce_lr],
    verbose=1
)

print("\n✅ Entrenamiento completado!")


In [ ]:
# Guardar modelo final
model.save('modelo_quemaduras_cortadas.keras')
print("✅ Modelo guardado como 'modelo_quemaduras_cortadas.keras'")


In [ ]:
# Graficar resultados del entrenamiento
plt.figure(figsize=(12, 4))

# Gráfico de precisión
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entrenamiento', marker='o')
plt.plot(history.history['val_accuracy'], label='Validación', marker='s')
plt.xlabel('Épocas')
plt.ylabel('Precisión')
plt.title('Precisión del Modelo')
plt.legend()
plt.grid(True)

# Gráfico de pérdida
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entrenamiento', marker='o')
plt.plot(history.history['val_loss'], label='Validación', marker='s')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.title('Pérdida del Modelo')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Mostrar mejores resultados
best_val_acc = max(history.history['val_accuracy'])
best_train_acc = max(history.history['accuracy'])
final_val_acc = history.history['val_accuracy'][-1]
final_train_acc = history.history['accuracy'][-1]

print(f"\n📊 Resultados del entrenamiento:")
print(f"   Mejor precisión de validación: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")
print(f"   Mejor precisión de entrenamiento: {best_train_acc:.4f} ({best_train_acc*100:.2f}%)")
print(f"   Precisión final de validación: {final_val_acc:.4f} ({final_val_acc*100:.2f}%)")
print(f"   Precisión final de entrenamiento: {final_train_acc:.4f} ({final_train_acc*100:.2f}%)")


In [ ]:
# Descargar modelo entrenado
from google.colab import files

# Opción 1: Descargar directamente
files.download('modelo_quemaduras_cortadas.keras')

# Opción 2: Guardar en Google Drive (descomenta si prefieres)
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('modelo_quemaduras_cortadas.keras', '/content/drive/MyDrive/')
# print("✅ Modelo guardado en Google Drive")

print("✅ Modelo descargado")
